# Brachistochrone-constrained cat preparation, per gate set

Prepare an even cat from vacuum with each gate set (ECD, SNAP, CSQ `single`, CSQ `ab`), using the geodesic penalty to keep each intermediate cavity state near the brachistochrone arc.

In [1]:
%load_ext autoreload
%autoreload 2

import jax
jax.config.update("jax_enable_x64", True)

import matplotlib.pyplot as plt
import jax.numpy as jnp
import jaxquantum as jqt

from gkp_optimal_control.gate_optimization import (
    optimize_gate_sequence, GateBounds, OptimizerConfig, TrajectoryPenalty, sequence_history
)
from gkp_optimal_control.states import cat_states

## Setup

In [2]:
n_fock = 30
alpha = 1.5

vac = jnp.zeros(n_fock, dtype=jnp.complex64).at[0].set(1.0)
cat, _ = cat_states(n_fock, alpha)  # even cat

geo = TrajectoryPenalty.notes_defaults()
opt = OptimizerConfig(n_seeds=256, n_adam_iters=1000, seed=0)
bounds = GateBounds(max_disp=3.0, max_squeeze=0.8, n_leak=4)

runs = {
    "ECD":        dict(gate_set="ecd",  n_gates=8, disp_method="quadrature"),
    "SNAP":       dict(gate_set="snap", n_gates=8, n_snap=10, disp_method="quadrature"),
    # "CSQ single": dict(gate_set="csq",  n_gates=8, layer="single"),
    # "CSQ ab":     dict(gate_set="csq",  n_gates=8, layer="ab"),
}

## Run

In [3]:
results = {}
for name, kw in runs.items():
    print(f"\n===== {name} =====")
    results[name] = optimize_gate_sequence(
        psi_init=vac, psi_targ=cat,
        loss_type="log_infidelity",
        bounds=bounds, optimizer=opt,
        **kw,
    )


===== ECD =====
gate set : ecd   depth : 8   n_fock : 30
params   : 34   state pairs : 1   reduction : mean
loss     : log_infidelity   disp : quadrature   seeds : 256
--------------------------------------------------------------



KeyboardInterrupt


KeyboardInterrupt



## Summary

In [ ]:
print(f"{'gate set':<12}{'F':>10}{'max geo dev':>14}{'max F drop':>13}{'qubit purity':>14}")
for name, r in results.items():
    drop = jnp.maximum(r.trajectory[:-1] - r.trajectory[1:], 0).max()
    pur = "-" if r.qubit_purity is None else f"{r.qubit_purity:.4f}"
    print(f"{name:<12}{r.fidelity:>10.5f}{r.geodesic['deviation'].max():>14.3e}{drop:>13.3e}{pur:>14}")

## Paths

The x-axis is the layer index normalized to [0, 1]. The dashed line in the middle panel is the brachistochrone's constant-speed schedule, and in the right panel it is the fidelity that schedule would give, cos²((1−s)θ_B).

In [ ]:
theta_b = jnp.arccos(abs(complex((jnp.transpose(vac) @ cat).squeeze())))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, r in results.items():
    s = jnp.linspace(0, 1, len(r.trajectory))
    axes[0].plot(s, r.geodesic["deviation"], "o-", label=name)
    axes[1].plot(s, r.geodesic["progress"], "o-", label=name)
    axes[2].plot(s, r.trajectory, "o-", label=name)

s_ref = jnp.linspace(0, 1, 100)
axes[1].plot(s_ref, s_ref, "k--", lw=1, label="brachistochrone")
axes[2].plot(s_ref, jnp.cos((1 - s_ref) * theta_b) ** 2, "k--", lw=1, label="brachistochrone")

axes[0].set(xlabel="layer / N", ylabel="1 − max f(a)", title="Distance to geodesic", yscale="log")
axes[1].set(xlabel="layer / N", ylabel="a* / θ_B", title="Progress along geodesic")
axes[2].set(xlabel="layer / N", ylabel="⟨cat|ρ_cav|cat⟩", title="Fidelity path")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
def path_efficiency(res, psi_init):
    h = sequence_history(res, psi_init)[:, 0]        # first state pair; states after every gate
    h = h.reshape(h.shape[0], -1)                    # flatten qubit blocks -> joint kets
    h = h / jnp.linalg.norm(h, axis=-1, keepdims=True)
    steps = jnp.arccos(jnp.clip(jnp.abs(jnp.sum(h[:-1].conj() * h[1:], -1)), 0, 1))
    theta_b = jnp.arccos(jnp.clip(abs(jnp.vdot(h[0], h[-1])), 0, 1))
    return theta_b / steps.sum(), steps

for name, r in results.items():
    eta, steps = path_efficiency(r, vac)
    print(f"{name:<12} η = {eta:.3f}   L = {steps.sum():.3f}")

Things to try next:
- Set `geodesic_weight=0` to see how far an unconstrained solution wanders.
- Sweep the weight, warm-starting each run from the previous one with `params0=prev.flat_params`.
- Switch to `mode="schedule"` to require constant Bures speed.